In [ ]:
!pip install pysindy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.4/123.4 kB 4.2 MB/s eta 0:00:00


In [ ]:
import numpy as np
import pysindy as ps

from scipy.integrate import solve_ivp
from itertools import combinations_with_replacement

import matplotlib.pyplot as plt
import pandas as pd
import time

In [ ]:
# ==========================================================
# CONFIGURATION
# ==========================================================

SIGMA = 10.0
BETA = 8.0 / 3.0
RHO = 28.0

X0 = [-8.0, 7.0, 27.0]

DT = 0.001
TSPAN = (0, 50)

DEFAULT_THRESHOLD = 0.1

DEGREES = [1,2,3,4,5,6,7]

NOISE_LEVELS = [
    0,
    0.001,
    0.005,
    0.01,
    0.05,
    0.1,
    0.5,
    1.0
]

In [ ]:
def lorenz(
    t,
    state,
    sigma=SIGMA,
    beta=BETA,
    rho=RHO
):
    x, y, z = state

    dx = sigma*(y-x)
    dy = x*(rho-z)-y
    dz = x*y-beta*z

    return [dx,dy,dz]

In [ ]:
def generate_lorenz_data():

    t_eval = np.arange(
        TSPAN[0],
        TSPAN[1],
        DT
    )

    sol = solve_ivp(
        lorenz,
        TSPAN,
        X0,
        t_eval=t_eval,
        method="RK45",
        rtol=1e-10,
        atol=1e-10
    )

    X = sol.y.T
    t = sol.t

    Xdot = np.zeros_like(X)

    for i in range(len(t)):
        Xdot[i] = lorenz(
            t[i],
            X[i]
        )

    return X, Xdot, t

In [ ]:
def build_library(X, degree):

    m,n = X.shape

    Theta = np.ones((m,1))
    labels = ["1"]

    variables = ["x","y","z"][:n]

    for d in range(1, degree+1):

        for combo in combinations_with_replacement(
            range(n),
            d
        ):

            col = np.ones(m)

            label = ""

            for idx in combo:
                col *= X[:,idx]
                label += variables[idx]

            Theta = np.column_stack(
                [Theta,col]
            )

            labels.append(label)

    return Theta, labels

In [ ]:
def stls(
    Theta,
    xdot,
    threshold,
    max_iter=100
):

    xi, *_ = np.linalg.lstsq(
        Theta,
        xdot,
        rcond=None
    )

    for _ in range(max_iter):

        small = np.abs(xi) < threshold

        xi[small] = 0

        active = ~small

        if np.sum(active) == 0:
            break

        xi_active, *_ = np.linalg.lstsq(
            Theta[:,active],
            xdot,
            rcond=None
        )

        xi[active] = xi_active

        if np.array_equal(
            small,
            np.abs(xi) < threshold
        ):
            break

    return xi


def run_sindy(
    Theta,
    Xdot,
    threshold=DEFAULT_THRESHOLD
):

    p = Theta.shape[1]
    n_eq = Xdot.shape[1]

    Xi = np.zeros((p,n_eq))

    for k in range(n_eq):

        Xi[:,k] = stls(
            Theta,
            Xdot[:,k],
            threshold
        )

    return Xi

In [ ]:
def add_noise(
    X,
    noise_level
):

    if noise_level == 0:
        return X.copy()

    return (
        X
        + noise_level*np.random.randn(*X.shape)
    )


def finite_diff(
    X,
    dt
):

    Xdot = np.zeros_like(X)

    Xdot[0] = (X[1]-X[0])/dt

    Xdot[1:-1] = (
        X[2:] - X[:-2]
    )/(2*dt)

    Xdot[-1] = (
        X[-1]-X[-2]
    )/dt

    return Xdot

In [ ]:
def get_true_coefficients(labels):

    Xi_true = np.zeros(
        (len(labels),3)
    )

    for i,label in enumerate(labels):

        if label == "x":
            Xi_true[i,0] = -10

        if label == "y":
            Xi_true[i,0] = 10

        if label == "x":
            Xi_true[i,1] = 28

        if label == "y":
            Xi_true[i,1] = -1

        if label == "xz":
            Xi_true[i,1] = -1

        if label == "z":
            Xi_true[i,2] = -8/3

        if label == "xy":
            Xi_true[i,2] = 1

    return Xi_true


def check_support(
    Xi_true,
    Xi_rec,
    tol=1e-6
):

    true_nz = np.abs(Xi_true) > tol
    rec_nz = np.abs(Xi_rec) > tol

    return np.array_equal(
        true_nz,
        rec_nz
    )


def coeff_error(
    Xi_true,
    Xi_rec
):

    return (
        np.linalg.norm(
            Xi_true-Xi_rec
        )
        /
        np.linalg.norm(Xi_true)
    )

In [ ]:
def mutual_coherence(Theta):

    Theta_norm = Theta / np.linalg.norm(
        Theta,
        axis=0,
        keepdims=True
    )

    G = np.abs(
        Theta_norm.T @ Theta_norm
    )

    np.fill_diagonal(G, 0)

    return np.max(G)

In [ ]:
# ==========================================================
# SINGULAR-VALUE DIAGNOSTICS
# ==========================================================
# sigma_min is the smallest singular value of the (column-scaled) library.
# It controls how ill-posed the least-squares solve inside STLS is:
#   - cond = sigma_max / sigma_min  measures the SPREAD of the spectrum
#   - sigma_min alone measures how close the library is to being
#     RANK-DEFICIENT, i.e. how close two or more candidate columns are
#     to being linearly dependent.
#
# We scale columns to unit norm first so that sigma_min reflects genuine
# (near-)collinearity among candidate functions rather than differences in
# the raw magnitude of, say, x vs x^6.

def singular_value_diagnostics(Theta, rank_tol=None):
    """
    Returns (sigma_min, sigma_max, numerical_rank, n_cols) for the
    column-normalised library Theta.

    numerical_rank counts singular values above rank_tol. By default
    rank_tol = sigma_max * max(shape) * eps, the standard numpy threshold.
    """
    # column-normalise (guard against zero-norm columns, e.g. a constant of 0)
    norms = np.linalg.norm(Theta, axis=0, keepdims=True)
    norms[norms == 0] = 1.0
    Theta_n = Theta / norms

    s = np.linalg.svd(Theta_n, compute_uv=False)

    sigma_max = s[0]
    sigma_min = s[-1]

    if rank_tol is None:
        rank_tol = sigma_max * max(Theta_n.shape) * np.finfo(float).eps

    numerical_rank = int(np.sum(s > rank_tol))

    return sigma_min, sigma_max, numerical_rank, Theta_n.shape[1]


## Why σ_min matters here, and what it explains about degree 6

The condition number `cond = σ_max / σ_min` summarises the **spread** of the library's
singular spectrum, but it conflates two different things: a large `σ_max` (some columns
have big magnitude) and a small `σ_min` (some columns are nearly linearly dependent).
For sparse *support* recovery it is the small end of the spectrum that does the damage,
so we log **σ_min** separately. We compute it on the **column-normalised** library, so a
small σ_min means genuine near-collinearity among candidate functions, not just a scale
difference between `x` and `x⁶`.

**σ_min is the distance to rank-deficiency.** When σ_min is near zero there is a direction
in candidate-function space along which the library is almost singular: two or more columns
are nearly collinear, and the least-squares step inside STLS has no well-defined answer
along that direction. The solver is free to distribute weight between the collinear columns
almost arbitrarily, and small changes in the data move that split by a lot. σ_min is a
*continuous* diagnostic and is the one to trust; the integer "numerical rank" we also log is
coarser because it depends on an arbitrary cutoff tolerance, so treat it as a sanity check
rather than the main signal.

**This is exactly what produces the degree-6 zero-noise failure.** Going from degree 2 to
degree 6 the column-normalised σ_min collapses by roughly four orders of magnitude
(≈ 2×10⁻² → ≈ 7×10⁻⁷). The failure is localised to the **z-equation**, whose true terms are
`z` and `xy`. The column `xy` is ~0.97 correlated with `yy` along the attractor, so the clean
least-squares solve splits the true `xy` weight across both columns — the trace cell below
shows `coeff(xy) ≈ 0.07`, which is *below* the STLS threshold of 0.1 and therefore gets
thresholded to zero. STLS then locks onto a spurious support `['1', 'z', 'yy']` that omits
`xy`.

Adding a whisker of noise (η = 0.001) lifts σ_min and perturbs the collinear split just
enough that the `xy` coefficient lands back **above** threshold, so STLS converges to the
correct support `['z', 'xy']`. So "noise helps at degree 6" is not noise being beneficial —
it is **noise breaking a numerical degeneracy** (dithering / implicit regularisation),
precisely the cautious reading your professor suggested. Once the noise grows past ~0.1 it
corrupts the finite-difference derivatives faster than it helps, and recovery collapses —
giving the inverted-U in noise.

The cells below add σ_min, σ_max, numerical rank, and rank deficiency to the sweep, plot
them against degree and against recovery probability, and run a focused trace of the
degree-6 z-equation.

> **Runtime knob.** The full 100-seed sweep is dominated by the SVD diagnostics on the
> large high-degree libraries. Because those diagnostics barely change across seeds, the
> experiment computes them on only the first `n_diag_seeds` (default 5) while still running
> STLS recovery on all 100 seeds — recovery probability is identical, the run is much faster.
> To reproduce the original exhaustive behaviour, call with `n_diag_seeds=100`.

In [ ]:
def recovery_probability_experiment(
    X,
    Xdot,
    dt,
    degrees,
    noise_levels,
    n_seeds=100,
    n_diag_seeds=5,
    threshold=0.1
):
    """
    Support recovery probability across random seeds.

    SPEED NOTE: the expensive part of each iteration is the SVD-based diagnostics
    (condition number, coherence, sigma_min) on a 50000 x p library. These barely
    vary from seed to seed for a fixed (degree, noise) -- the std columns in earlier
    runs were tiny relative to the means. So we compute the diagnostics on only the
    first `n_diag_seeds` seeds, while still running the cheap STLS recovery on all
    `n_seeds`. Recovery probability is therefore unchanged; the diagnostic means are
    estimated from a handful of seeds instead of 100, which cuts a large chunk of the
    runtime. Set n_diag_seeds = n_seeds to recover the original exhaustive behaviour.

    Returns a DataFrame with, per (degree, noise):
        degree, noise, library_size, recovery_probability,
        mean_condition / std_condition, mean_coherence / std_coherence,
        mean_sigma_min / std_sigma_min, mean_sigma_max,
        mean_numerical_rank, mean_rank_deficiency, n_diag_seeds
    """

    results = []

    for deg in degrees:

        print(f"Degree {deg}")

        for noise in noise_levels:

            successes = 0

            cond_numbers = []
            coherences = []
            sigma_mins = []
            sigma_maxs = []
            ranks = []

            for seed in range(n_seeds):

                np.random.seed(seed)

                X_noisy = add_noise(X, noise)

                if noise == 0:
                    Xdot_used = Xdot
                else:
                    Xdot_used = finite_diff(X_noisy, dt)

                Theta, labels = build_library(X_noisy, deg)

                # --- expensive diagnostics: only on the first n_diag_seeds ---
                if seed < n_diag_seeds:
                    cond_numbers.append(np.linalg.cond(Theta))
                    coherences.append(mutual_coherence(Theta))
                    s_min, s_max, n_rank, n_cols = singular_value_diagnostics(Theta)
                    sigma_mins.append(s_min)
                    sigma_maxs.append(s_max)
                    ranks.append(n_rank)

                # --- cheap recovery: on all n_seeds ---
                Xi_rec = run_sindy(Theta, Xdot_used, threshold)
                Xi_true = get_true_coefficients(labels)

                if check_support(Xi_true, Xi_rec):
                    successes += 1

            library_size = Theta.shape[1]
            mean_rank = np.mean(ranks)

            results.append({
                "degree": deg,
                "noise": noise,
                "library_size": library_size,

                "recovery_probability": successes / n_seeds,

                "mean_condition": np.mean(cond_numbers),
                "std_condition": np.std(cond_numbers),

                "mean_coherence": np.mean(coherences),
                "std_coherence": np.std(coherences),

                "mean_sigma_min": np.mean(sigma_mins),
                "std_sigma_min": np.std(sigma_mins),
                "mean_sigma_max": np.mean(sigma_maxs),
                "mean_numerical_rank": mean_rank,
                "mean_rank_deficiency": library_size - mean_rank,

                "n_diag_seeds": n_diag_seeds,
            })

    return pd.DataFrame(results)

In [ ]:
X, Xdot, t = generate_lorenz_data()

df = recovery_probability_experiment(
    X=X,
    Xdot=Xdot,
    dt=DT,
    degrees=DEGREES,
    noise_levels=NOISE_LEVELS,
    n_seeds=100,
    threshold=0.1
)

df

Degree 1
Degree 2
Degree 3
Degree 4
Degree 5
Degree 6
Degree 7


,degree,noise,library_size,recovery_probability,mean_condition,std_condition,mean_coherence,std_coherence
0,1,0.000,4,0.00,7.227314e+01,2.842171e-14,0.937717,2.220446e-16
1,1,0.001,4,0.00,7.227314e+01,3.950627e-05,0.937717,6.623286e-08
2,1,0.005,4,0.00,7.227312e+01,1.975347e-04,0.937717,3.311722e-07
3,1,0.010,4,0.00,7.227309e+01,3.950776e-04,0.937717,6.623644e-07
4,1,0.050,4,0.00,7.227216e+01,1.975660e-03,0.937715,3.312629e-06
5,1,0.100,4,0.00,7.226935e+01,3.951734e-03,0.937709,6.627326e-06
6,1,0.500,4,0.00,7.218085e+01,1.972216e-02,0.937528,3.322957e-05
7,1,1.000,4,0.00,7.190843e+01,3.909709e-02,0.936966,6.674167e-05
8,2,0.000,10,1.00,9.482411e+03,0.000000e+00,0.970077,2.220446e-16
9,2,0.001,10,1.00,9.482392e+03,6.144108e-02,0.970077,1.618631e-07


In [ ]:
df.to_csv(
    "lorenz_recovery_results.csv",
    index=False
)

---
# Visualisations

A clean, consistent figure suite. Run the **style** cell first, then each plot
cell. Figures are grouped into: (1) the headline recovery summary, (2) recovery
vs degree and vs noise, (3) library-conditioning diagnostics vs degree, (4) the
mechanistic recovery-vs-diagnostic scatters, and (5) the degree-6 anomaly trace.

In [ ]:
# ==========================================================
# PLOTTING STYLE  (run once; applies to every figure below)
# ==========================================================
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

plt.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 120, "savefig.bbox": "tight",
    "font.size": 11, "font.family": "DejaVu Sans",
    "axes.titlesize": 13, "axes.titleweight": "bold", "axes.titlepad": 12,
    "axes.labelsize": 11, "axes.labelpad": 6,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.linewidth": 0.9, "axes.edgecolor": "#444444",
    "axes.grid": True, "grid.alpha": 0.22, "grid.linewidth": 0.6, "grid.color": "#888888",
    "legend.frameon": False, "legend.fontsize": 9,
    "xtick.color": "#333333", "ytick.color": "#333333",
    "figure.facecolor": "white", "axes.facecolor": "white",
    "lines.linewidth": 2.0, "lines.markersize": 6,
})

SYSTEM_NAME = "Lorenz"
RECOV_CMAP = LinearSegmentedColormap.from_list("recov", ["#d73027", "#fee08b", "#1a9850"])

def _noise_label(nl):
    return "η = 0" if nl == 0 else f"η = {nl:g}"

def noise_colors(noise_levels):
    cmap = plt.cm.viridis
    n = len(noise_levels)
    return {nl: cmap(0.08 + 0.84*i/(max(n-1,1))) for i, nl in enumerate(sorted(noise_levels))}

def degree_colors(degrees):
    cmap = plt.cm.plasma
    n = len(degrees)
    return {d: cmap(0.05 + 0.85*i/(max(n-1,1))) for i, d in enumerate(sorted(degrees))}


### 1 · Headline: recovery probability heatmap

In [ ]:
# ---- Headline plot: recovery probability heatmap (degree x noise) ----
import numpy as np
degs = sorted(df.degree.unique()); noises = sorted(df.noise.unique())
M = np.full((len(noises), len(degs)), np.nan)
for i, nl in enumerate(noises):
    for j, d in enumerate(degs):
        v = df[(df.noise == nl) & (df.degree == d)].recovery_probability
        if len(v): M[i, j] = v.values[0]

fig, ax = plt.subplots(figsize=(8.6, 5.4))
im = ax.imshow(M, cmap=RECOV_CMAP, vmin=0, vmax=1, aspect="auto", origin="upper")
ax.set_xticks(range(len(degs))); ax.set_xticklabels([f"deg {d}" for d in degs])
ax.set_yticks(range(len(noises))); ax.set_yticklabels([_noise_label(n) for n in noises])
for i in range(len(noises)):
    for j in range(len(degs)):
        val = M[i, j]
        if not np.isnan(val):
            ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                    color="white" if (val < 0.28 or val > 0.72) else "#222222",
                    fontsize=9, fontweight="bold")
ax.set_xlabel("Polynomial library degree"); ax.set_ylabel("Noise level η")
ax.set_title(f"{SYSTEM_NAME}: Support Recovery Probability")
cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04); cb.set_label("Recovery probability")
ax.grid(False)
fig.tight_layout(); plt.show()


### 2 · Recovery probability vs degree and vs noise

In [ ]:
# ---- Recovery probability vs degree (one line per noise) ----
nc = noise_colors(df.noise.unique())
fig, ax = plt.subplots(figsize=(8.6, 5.4))
for nl in sorted(df.noise.unique()):
    s = df[df.noise == nl].sort_values("degree")
    ax.plot(s.degree, 100*s.recovery_probability, marker="o", color=nc[nl], label=_noise_label(nl))
ax.set_xlabel("Polynomial library degree"); ax.set_ylabel("Recovery probability (%)")
ax.set_title(f"{SYSTEM_NAME}: Recovery Probability vs Library Degree")
ax.set_ylim(-4, 104)
ax.legend(title="Noise", ncol=2, loc="lower left")
fig.tight_layout(); plt.show()


In [ ]:
# ---- Recovery probability vs noise (one line per degree): the ANOMALY view ----
# eta=0 is plotted at the far left on a log axis. A line that JUMPS UP from
# eta=0 to the first nonzero noise is the "noise breaks a degeneracy" anomaly.
dc = degree_colors(df.degree.unique())
pos_min = df[df.noise > 0].noise.min()
fig, ax = plt.subplots(figsize=(8.6, 5.4))
for d in sorted(df.degree.unique()):
    s = df[df.degree == d].sort_values("noise").copy()
    s["xn"] = s.noise.replace(0, pos_min/3)
    ax.plot(s.xn, 100*s.recovery_probability, marker="o", color=dc[d],
            label=f"deg {d}", alpha=0.9)
ax.set_xscale("log")
ax.set_xlabel("Noise level η   (η = 0 shown at far left)")
ax.set_ylabel("Recovery probability (%)")
ax.set_title(f"{SYSTEM_NAME}: Recovery Probability vs Noise  (anomaly view)")
ax.set_ylim(-4, 108)
ax.legend(title="Degree", ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.16))
fig.tight_layout(); plt.show()


### 3 · Library-conditioning diagnostics vs degree

In [ ]:
# ---- Condition number vs degree (log scale, one line per noise) ----
nc = noise_colors(df.noise.unique())
fig, ax = plt.subplots(figsize=(8.6, 5.4))
for nl in sorted(df.noise.unique()):
    s = df[df.noise == nl].sort_values("degree")
    ax.semilogy(s.degree, s.mean_condition, marker="o", color=nc[nl], label=_noise_label(nl))
ax.set_xlabel("Polynomial library degree"); ax.set_ylabel("Condition number κ(Θ)")
ax.set_title(f"{SYSTEM_NAME}: Library Conditioning vs Degree")
ax.legend(title="Noise", ncol=2)
fig.tight_layout(); plt.show()


In [ ]:
# ---- Condition number growth law: log10(kappa) vs degree, linear fit (eta=0) ----
import numpy as np
s = df[df.noise == 0].sort_values("degree")
d = s.degree.values.astype(float); logk = np.log10(s.mean_condition.values)
slope, intercept = np.polyfit(d, logk, 1)
fig, ax = plt.subplots(figsize=(8.6, 5.4))
ax.plot(d, logk, "o", color="#2166ac", markersize=8, label="measured (η=0)")
dd = np.linspace(d.min(), d.max(), 100)
ax.plot(dd, slope*dd + intercept, "--", color="#b2182b",
        label=f"fit:  log₁₀κ ≈ {slope:.2f}·d + {intercept:.2f}")
ax.set_xlabel("Polynomial library degree d"); ax.set_ylabel("log₁₀ κ(Θ)")
ax.set_title(f"{SYSTEM_NAME}: Condition Number Growth Law")
ax.legend(loc="upper left")
fig.tight_layout(); plt.show()
print(f"Growth law: kappa ~ 10^({slope:.2f} d), i.e. multiply by ~{10**slope:.0f}x per degree")


In [ ]:
# ---- Mutual coherence vs degree ----
nc = noise_colors(df.noise.unique())
fig, ax = plt.subplots(figsize=(8.6, 5.4))
for nl in sorted(df.noise.unique()):
    s = df[df.noise == nl].sort_values("degree")
    ax.plot(s.degree, s.mean_coherence, marker="o", color=nc[nl], label=_noise_label(nl))
ax.set_xlabel("Polynomial library degree"); ax.set_ylabel("Mutual coherence μ(Θ)")
ax.set_title(f"{SYSTEM_NAME}: Mutual Coherence vs Degree")
ax.legend(title="Noise", ncol=2, loc="lower right")
fig.tight_layout(); plt.show()


In [ ]:
# ---- Smallest singular value sigma_min vs degree (log scale) ----
nc = noise_colors(df.noise.unique())
fig, ax = plt.subplots(figsize=(8.6, 5.4))
for nl in sorted(df.noise.unique()):
    s = df[df.noise == nl].sort_values("degree")
    ax.semilogy(s.degree, s.mean_sigma_min, marker="o", color=nc[nl], label=_noise_label(nl))
ax.set_xlabel("Polynomial library degree"); ax.set_ylabel("σ_min  (column-normalised Θ)")
ax.set_title(f"{SYSTEM_NAME}: Onset of Near-Rank-Deficiency vs Degree")
ax.legend(title="Noise", ncol=2)
fig.tight_layout(); plt.show()


In [ ]:
# ---- Three library diagnostics side by side (eta=0) ----
s = df[df.noise == 0].sort_values("degree")
fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
axes[0].semilogy(s.degree, s.mean_condition, "o-", color="#2166ac")
axes[0].set_title("Condition number κ"); axes[0].set_xlabel("degree"); axes[0].set_ylabel("κ(Θ)")
axes[1].plot(s.degree, s.mean_coherence, "o-", color="#5aae61")
axes[1].set_title("Mutual coherence μ"); axes[1].set_xlabel("degree"); axes[1].set_ylabel("μ(Θ)")
axes[2].semilogy(s.degree, s.mean_sigma_min, "o-", color="#b2182b")
axes[2].set_title("Smallest singular value σ_min"); axes[2].set_xlabel("degree"); axes[2].set_ylabel("σ_min")
fig.suptitle(f"{SYSTEM_NAME}: Library Diagnostics vs Degree (η=0)", fontsize=14, fontweight="bold", y=1.04)
fig.tight_layout(); plt.show()


### 4 · Mechanistic links: recovery vs each diagnostic

The key comparison. On Lorenz, coherence is saturated near 1 and barely varies, so
its scatter is a vertical smear; condition number and σ_min span many orders of
magnitude and separate success from failure far better. This is the visual evidence
that **κ and σ_min are the informative predictors and coherence is blunt here.**

In [ ]:
# ---- Recovery probability vs each diagnostic (the mechanistic links) ----
# If recovery collapses onto a clean curve against one diagnostic, that
# diagnostic is the better predictor. Compare the spread on each x-axis.
degs = sorted(df.degree.unique())
fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
specs = [("mean_condition", "Condition number κ", True),
         ("mean_sigma_min", "σ_min", True),
         ("mean_coherence", "Mutual coherence μ", False)]
for ax, (col, lab, logx) in zip(axes, specs):
    sc = ax.scatter(df[col], 100*df.recovery_probability, c=df.degree, cmap="plasma",
                    s=55, edgecolor="white", linewidth=0.5, vmin=min(degs), vmax=max(degs))
    if logx: ax.set_xscale("log")
    ax.set_xlabel(lab); ax.set_ylabel("Recovery probability (%)"); ax.set_ylim(-4, 104)
    ax.set_title(f"vs {lab}")
cb = fig.colorbar(sc, ax=axes, fraction=0.025, pad=0.02); cb.set_label("degree")
fig.suptitle(f"{SYSTEM_NAME}: Recovery Probability vs Library Diagnostics",
             fontsize=14, fontweight="bold", y=1.04)
plt.show()


In [ ]:
# ==========================================================
# DEGREE-6 ANOMALY: focused diagnostic
# ==========================================================
# At degree 6, recovery is 0.00 at zero noise but ~1.00 at tiny noise.
# This cell traces the mechanism for a single clean vs tiny-noise case
# by inspecting the FAILING equation directly.
#
# Result you should see: in the clean library the z-equation least-squares
# solve splits the true xy weight between the highly correlated columns
# xy and yy. The xy coefficient is pushed BELOW the threshold and killed,
# so STLS converges to a spurious support. A whisker of noise breaks the
# near-degeneracy and the xy coefficient lands back above threshold.

deg = 6

# clean library
Theta_c, labels_c = build_library(X, deg)

# the z-equation is index 2; its true support is {z, xy}
xi_z_clean, *_ = np.linalg.lstsq(Theta_c, Xdot[:,2], rcond=None)

i_xy = labels_c.index("xy")
i_yy = labels_c.index("yy")

# correlation of the two competing columns along the trajectory
xy_col = Theta_c[:, i_xy]
yy_col = Theta_c[:, i_yy]
corr = abs(
    (xy_col/np.linalg.norm(xy_col)) @ (yy_col/np.linalg.norm(yy_col))
)

print("Degree-6 z-equation, CLEAN initial least-squares solve")
print(f"  corr(xy, yy) along attractor = {corr:.4f}")
print(f"  coeff(xy) = {xi_z_clean[i_xy]:+.4f}   (true value +1.0)")
print(f"  coeff(yy) = {xi_z_clean[i_yy]:+.4f}   (true value  0.0)")
print(f"  threshold = {DEFAULT_THRESHOLD}")
print(f"  -> xy is {'BELOW' if abs(xi_z_clean[i_xy])<DEFAULT_THRESHOLD else 'above'} threshold "
      f"and will be {'KILLED' if abs(xi_z_clean[i_xy])<DEFAULT_THRESHOLD else 'kept'} by STLS")

# recovered support clean vs tiny noise
s_min_c, s_max_c, rank_c, ncol_c = singular_value_diagnostics(Theta_c)
print(f"\n  clean library: sigma_min={s_min_c:.3e}, "
      f"numerical_rank={rank_c}/{ncol_c}, rank_deficiency={ncol_c-rank_c}")

np.random.seed(0)
X_tiny = add_noise(X, 0.001)
Xdot_tiny = finite_diff(X_tiny, DT)
Theta_t, labels_t = build_library(X_tiny, deg)
xi_z_tiny = stls(Theta_t, Xdot_tiny[:,2], DEFAULT_THRESHOLD)
sup_tiny = [labels_t[i] for i in range(len(labels_t)) if abs(xi_z_tiny[i])>1e-6]

xi_z_clean_stls = stls(Theta_c, Xdot[:,2], DEFAULT_THRESHOLD)
sup_clean = [labels_c[i] for i in range(len(labels_c)) if abs(xi_z_clean_stls[i])>1e-6]

print(f"\n  z-equation recovered support, CLEAN      : {sup_clean}")
print(f"  z-equation recovered support, η=0.001    : {sup_tiny}")
print(f"  true support                             : ['z', 'xy']")


### 5 · The degree-6 anomaly, visualised

The plot below traces the **initial least-squares coefficients of the z-equation**
(true terms `z`, `xy`) as noise increases. At η=0 the true `xy` coefficient sits
*below* the STLS threshold (in the shaded kill-zone) because its weight has leaked
into the highly-correlated `yy` column — so STLS deletes it and converges to the
wrong support. A whisker of noise breaks this degeneracy and pushes `xy` back above
threshold, restoring recovery. This is the mechanism behind the "noise helps at
degree 6" anomaly.

In [ ]:
# ---- Degree-6 anomaly: z-equation threshold-crossing plot ----
deg_anom = 6
THR = DEFAULT_THRESHOLD
noise_grid = [0, 0.0005, 0.001, 0.002, 0.005, 0.01, 0.02, 0.05]

xy_mag, yy_mag = [], []
for nl in noise_grid:
    np.random.seed(0)
    if nl == 0:
        Xn, Xd = X, Xdot
    else:
        Xn = add_noise(X, nl)
        Xd = finite_diff(Xn, DT)
    Theta_a, labels_a = build_library(Xn, deg_anom)
    xi_z, *_ = np.linalg.lstsq(Theta_a, Xd[:, 2], rcond=None)  # z-equation
    xy_mag.append(abs(xi_z[labels_a.index("xy")]))
    yy_mag.append(abs(xi_z[labels_a.index("yy")]))

pos_min = min(n for n in noise_grid if n > 0)
xgrid = [n if n > 0 else pos_min/3 for n in noise_grid]

fig, ax = plt.subplots(figsize=(8.6, 5.4))
ax.plot(xgrid, xy_mag, "o-", color="#1a9850", label="|coeff(xy)|  — true term")
ax.plot(xgrid, yy_mag, "s-", color="#762a83", label="|coeff(yy)|  — spurious competitor")
ax.axhline(THR, ls="--", color="#b2182b", lw=1.6, label=f"STLS threshold λ = {THR}")
ax.fill_between(xgrid, 0, THR, color="#b2182b", alpha=0.06)
ax.set_xscale("log")
ax.annotate("xy below threshold\n→ deleted → wrong support",
            xy=(xgrid[0], xy_mag[0]), xytext=(xgrid[0]*1.25, 0.32), fontsize=9,
            arrowprops=dict(arrowstyle="->", color="#555555"))
ax.set_xlabel("Noise level η   (η = 0 shown at far left)")
ax.set_ylabel("Initial least-squares coefficient magnitude")
ax.set_title(f"{SYSTEM_NAME} Degree-6 Anomaly: z-equation threshold crossing")
ax.legend(loc="upper left")
fig.tight_layout(); plt.show()
